In [ ]:
#pip install notebook requests beautifulsoup4 pandas lxml scrapy

##1- Fast dataset creation **(Requests + BeautifulSoup)** — inside Notebook

This is the fastest method for small/medium sites and perfect for learning.

Perform Web Scraping on the following website:
 https://books.toscrape.com/catalogue/page-1.html  

Cell 1 — **imports**

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [10]:
base_url = "https://books.toscrape.com/catalogue/page-{}.html"
books_data = []

Cell 3 — **scrape ALL pages automatically**

In [11]:
for page in range(1, 51):
    print(f"Scraping Page {page}...")
    
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    books = soup.find_all("article", class_="product_pod")
    
    for book in books:
        
        
        book_name = book.h3.a["title"]
        
        
        price = book.find("p", class_="price_color").text.strip()
        
        
        stock = book.find("p", class_="instock availability").text.strip()
        
        
        relative_link = book.h3.a["href"]
        book_link = "https://books.toscrape.com/catalogue/" + relative_link.replace("../", "")
        
        
        book_response = requests.get(book_link)
        book_soup = BeautifulSoup(book_response.text, "html.parser")
        
        
        description_tag = book_soup.find("meta", attrs={"name": "description"})
        
        if description_tag:
            description = description_tag["content"].strip()
        else:
            description = "No description"
        
        books_data.append({
            "book_name": book_name,
            "book_description": description,
            "book_price": price,
            "stock": stock,
            "page": page
        })

Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Scraping Page 4...
Scraping Page 5...
Scraping Page 6...
Scraping Page 7...
Scraping Page 8...
Scraping Page 9...
Scraping Page 10...
Scraping Page 11...
Scraping Page 12...
Scraping Page 13...
Scraping Page 14...
Scraping Page 15...
Scraping Page 16...
Scraping Page 17...
Scraping Page 18...
Scraping Page 19...
Scraping Page 20...
Scraping Page 21...
Scraping Page 22...
Scraping Page 23...
Scraping Page 24...
Scraping Page 25...
Scraping Page 26...
Scraping Page 27...
Scraping Page 28...
Scraping Page 29...
Scraping Page 30...
Scraping Page 31...
Scraping Page 32...
Scraping Page 33...
Scraping Page 34...
Scraping Page 35...
Scraping Page 36...
Scraping Page 37...
Scraping Page 38...
Scraping Page 39...
Scraping Page 40...
Scraping Page 41...
Scraping Page 42...
Scraping Page 43...
Scraping Page 44...
Scraping Page 45...
Scraping Page 46...
Scraping Page 47...
Scraping Page 48...
Scraping Page 49...
Scraping Page 50...


Cell 4 — **convert to dataset**

In [20]:
df = pd.DataFrame(books_data)
df

,book_name,book_description,book_price,stock,page
0,A Light in the Attic,It's hard to imagine a world without A Light i...,Â£51.77,In stock,1
1,Tipping the Velvet,"""Erotic and absorbing...Written with starling ...",Â£53.74,In stock,1
2,Soumission,"Dans une France assez proche de la nÃ´tre, un ...",Â£50.10,In stock,1
3,Sharp Objects,"WICKED above her hipbone, GIRL across her hear...",Â£47.82,In stock,1
4,Sapiens: A Brief History of Humankind,From a renowned historian comes a groundbreaki...,Â£54.23,In stock,1
...,...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,,Â£55.53,In stock,50
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",High school student Kei Nagai is struck dead i...,Â£57.06,In stock,50
997,A Spy's Devotion (The Regency Spies of London #1),"In Englandâs Regency era, manners and elegan...",Â£16.97,In stock,50
998,1st to Die (Women's Murder Club #1),"James Patterson, bestselling author of the Ale...",Â£53.98,In stock,50


Cell 5 — **export CSV (FINAL DATASET)**

In [13]:
df.to_csv("books_data.csv", index=False)

In [14]:
df.shape

(1000, 5)

You now created a **real dataset from the web in under 3 minutes.**

1- **Create Scrapy Project**

In [15]:
!scrapy startproject quotes_project

'scrapy' is not recognized as an internal or external command,
operable program or batch file.


2- **Move Into Project Folder**

In [16]:
%cd quotes_project

[WinError 2] The system cannot find the file specified: 'quotes_project'
d:\HUN\3.2\deeep\Section 2\for-testing


In [22]:
pip install scrapy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
pip show scrapy

Name: Scrapy
Version: 2.14.1
Summary: A high-level Web Crawling and Web Scraping framework
Home-page: https://scrapy.org/
Author: 
Author-email: Scrapy developers <pablo@pablohoffman.com>
License-Expression: BSD-3-Clause
Location: d:\py video\Lib\site-packages
Requires: cryptography, cssselect, defusedxml, itemadapter, itemloaders, lxml, packaging, parsel, protego, pydispatcher, pyopenssl, queuelib, service-identity, tldextract, twisted, w3lib, zope-interface
Required-by: 
Note: you may need to restart the kernel to use updated packages.


3- **Create Spider**

In [26]:
!scrapy genspider books books.toscrape.com

'scrapy' is not recognized as an internal or external command,
operable program or batch file.


4- **Replace Spider Code**

In [18]:
import scrapy

class BooksSpider(scrapy.Spider):
    name = "books"
    allowed_domains = ["books.toscrape.com"]
    
    start_urls = [
        f"https://books.toscrape.com/catalogue/page-{i}.html"
        for i in range(1, 51)
    ]

    def parse(self, response):
        books = response.css("article.product_pod")
        
        for book in books:
            
            book_name = book.css("h3 a::attr(title)").get()
            price = book.css("p.price_color::text").get()
            stock = book.css("p.instock.availability::text").getall()
            stock = "".join(stock).strip()
            
            book_link = book.css("h3 a::attr(href)").get()
            book_link = response.urljoin(book_link)
            
            yield scrapy.Request(
                book_link,
                callback=self.parse_book,
                meta={
                    "book_name": book_name,
                    "price": price,
                    "stock": stock,
                    "page": response.url.split("-")[-1].split(".")[0]
                }
            )

    def parse_book(self, response):
        description = response.css("meta[name='description']::attr(content)").get()
        
        yield {
            "book_name": response.meta["book_name"],
            "book_description": description,
            "book_price": response.meta["price"],
            "stock": response.meta["stock"],
            "page": response.meta["page"]
        }

5- **Run Spider & Export CSV**

In [19]:
!scrapy crawl books -o books_scrapy.csv

'scrapy' is not recognized as an internal or external command,
operable program or batch file.
